# Optuna: пробуем библиотеку

Не пункт чеклиста, отдельный интерес - посмотреть, как Optuna ищет гиперпараметры по сравнению с ручным перебором через ParameterGrid (tune_hyperparameters в validation.py). Сравним на двух моделях из classic_models_exploration.ipynb: XGBoost (там перебор был по сетке 3x3x3x2x2=108 комбинаций) и CatBoost (там перебор был медленным, 2x3x2x2=24 комбинации, но каждая дольше). Препроцессинг тот же зафиксированный пайплайн

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

from preprocessing import preprocess_data_advanced
from validation import cross_validate_model, tune_hyperparameters

In [2]:
train_data = pd.read_csv("data/train.csv")
test_data = pd.read_csv("data/test.csv")
test_passenger_ids = test_data['PassengerId']

train_data, artifacts = preprocess_data_advanced(train_data, is_train=True)
test_data = preprocess_data_advanced(test_data, is_train=False, artifacts=artifacts)

categorical_cols = ['Pclass', 'Embarked', 'Title']
train_data = pd.get_dummies(train_data, columns=categorical_cols)
test_data = pd.get_dummies(test_data, columns=categorical_cols)
test_data = test_data.reindex(columns=train_data.drop('Survived', axis=1).columns, fill_value=0)

scale_cols = ['Age', 'Fare', 'SibSp', 'Parch', 'TicketGroupSize']

scaler = StandardScaler()
train_data[scale_cols] = scaler.fit_transform(train_data[scale_cols])
test_data[scale_cols] = scaler.transform(test_data[scale_cols])

X_train = train_data.drop(['Survived'], axis=1)
y_train = train_data['Survived']
X_test = test_data

X_train.shape, X_test.shape

((891, 18), (418, 18))

## XGBoost: ручной grid search vs Optuna

В classic_models_exploration.ipynb ручной перебор по сетке n_estimators/max_depth/learning_rate/subsample/colsample_bytree (108 комбинаций, полный перебор) дал 0.84622. Зададим Optuna то же пространство параметров, но как диапазоны, и дадим ей 50 попыток вместо полного перебора 108

In [3]:
import optuna
from xgboost import XGBClassifier

# WARNING глушит лог по каждому trial, иначе 50 строк вывода
optuna.logging.set_verbosity(optuna.logging.WARNING)


def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 300, step=50),
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
    }
    model = XGBClassifier(random_state=42, **params)
    scores = cross_validate_model(model, X_train, y_train)
    return scores.mean()


xgb_study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
xgb_study.optimize(xgb_objective, n_trials=50)

print(f"optuna: лучший скор {xgb_study.best_value:.5f}, параметры {xgb_study.best_params}")
print("ручной grid search: 0.84622, параметры {'colsample_bytree': 0.7, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 300, 'subsample': 1.0}")

optuna: лучший скор 0.84734, параметры {'n_estimators': 100, 'max_depth': 7, 'learning_rate': 0.05343320426294466, 'subsample': 0.7166030698623393, 'colsample_bytree': 0.922821233850875}
ручной grid search: 0.84622, параметры {'colsample_bytree': 0.7, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 300, 'subsample': 1.0}


Optuna нашла 0.84734 против 0.84622 у ручного grid search, за 50 попыток вместо полного перебора 108 комбинаций. Сама комбинация max_depth=7 и n_estimators=100 была и в ручной сетке (перебор полный), но там learning_rate/subsample/colsample_bytree брались только из фиксированного списка значений. Optuna подбирает их как непрерывные диапазоны (learning_rate=0.0534, subsample=0.717, colsample_bytree=0.923 - ни одно из этих чисел не входило в ручную сетку), и именно в этом её реальное преимущество здесь, а не в количестве попыток

## CatBoost: ручной grid search vs Optuna

CatBoost в classic_models_exploration.ipynb был самым медленным на переборе (24 комбинации, но каждая существенно дольше XGBoost/LightGBM), ручной перебор дал 0.84398. Здесь Optuna должна показать преимущество нагляднее: TPE-сэмплер сходится к хорошей области, не перебирая весь grid

In [4]:
from catboost import CatBoostClassifier


def catboost_objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 300, step=50),
        'depth': trial.suggest_int('depth', 4, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 5),
    }
    model = CatBoostClassifier(
        random_state=42, verbose=False, allow_writing_files=False, thread_count=-1, **params
    )
    scores = cross_validate_model(model, X_train, y_train)
    return scores.mean()


catboost_study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
catboost_study.optimize(catboost_objective, n_trials=25)

print(f"optuna: лучший скор {catboost_study.best_value:.5f}, параметры {catboost_study.best_params}")
print("ручной grid search: 0.84398, параметры {'depth': 4, 'iterations': 300, 'l2_leaf_reg': 1, 'learning_rate': 0.03}")

optuna: лучший скор 0.84398, параметры {'iterations': 300, 'depth': 5, 'learning_rate': 0.04107559866252153, 'l2_leaf_reg': 1.988402052997287}
ручной grid search: 0.84398, параметры {'depth': 4, 'iterations': 300, 'l2_leaf_reg': 1, 'learning_rate': 0.03}


Optuna сошлась к тому же скору 0.84398, что и ручной перебор, но с другими параметрами (depth=5 вместо 4, learning_rate=0.041 вместо 0.03), то есть нашла другую точку с тем же качеством. Здесь непрерывный поиск не дал прироста, похоже, для этой задачи и модели 0.844 - потолок при таком наборе фичей, а не следствие грубости сетки

## Итог

Optuna удобнее ручного ParameterGrid тем, что не требует заранее фиксировать сетку значений - можно задать диапазон и число попыток, и TPE-сэмплер сам решает, где искать дальше, опираясь на предыдущие результаты (а не перебирает все комбинации подряд, как ParameterGrid). На XGBoost это дало реальный прирост (0.847 против 0.846) за счёт непрерывного подбора learning_rate/subsample/colsample_bytree. На CatBoost прироста не дало, скор упёрся в тот же потолок 0.844 при других параметрах. Общий лидер по классическим моделям (XGBoost) не изменился, менять его параметры в основном пайплайне на найденные Optuna не будем - разница 0.001 внутри шума CV